# Conjunto de list partido por cuánto del oro tiene nodo en el grafo



In [1]:
import json
import re
import sys

import pandas as pd

sys.path.insert(0, "../../../graphrag")
from entidades.linkear_entidades import Linkeador, menciones_de

BIOASQ = "../../preguntas/BioASQ-training14b/training14b.json"
EXTRACCIONES = "extracciones.jsonl"
TIPOS_DE_NODO = ["Gene", "Disease", "Drug"]
#el nombre de la tarea en los scripts de la carpeta de arriba
TAREA = "list_por_oro"
GRUPOS = ["nada", "menos_de_la_mitad", "la_mitad_o_mas"]
#cuantas se sortean de cada grupo. no puede pasar del grupo mas chico (ver cuanto se va perdiendo)
#con la misma SEED, el sorteo de un numero mas grande contiene al de uno mas chico: subirlo despues suma preguntas
#sin cambiar las que ya se corrieron
PREGUNTAS_POR_GRUPO = 25
SEED = 42
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 100)

## Carga de las preguntas de list

In [2]:
with open(BIOASQ) as archivo:
    todas = json.load(archivo)["questions"]

preguntas = pd.DataFrame([{"id": p["id"], "body": p["body"], "oro": p["exact_answer"], "ideal_answer": p["ideal_answer"]}
                          for p in todas if p["type"] == "list"])
len(preguntas)

1130

## entidades de la pregunta

Igual que en armar_conjuntos.ipynb: las menciones guardadas de Qwen, cada una enlazada con linkear_mencion.

In [3]:
extracciones = {}
with open(EXTRACCIONES) as archivo:
    for linea in archivo:
        fila = json.loads(linea)
        extracciones[fila["id_bioasq"]] = fila["respuesta"]
assert preguntas["id"].isin(extracciones).all()

linkeador = Linkeador()

entidades, enlaces, sin_enlazar, menciones = [], [], [], []
for pregunta in preguntas.itertuples():
    entidades_de_la_pregunta, enlaces_de_la_pregunta, menciones_sin_nodo, textos = [], [], [], []
    for mencion in menciones_de(extracciones[pregunta.id]):
        textos.append(mencion.texto)
        enlace = linkeador.linkear_mencion(mencion)
        if enlace is None:
            menciones_sin_nodo.append(mencion.texto)
            continue
        label, ids, paso = enlace
        enlaces_de_la_pregunta.append([mencion.texto, label, ids, paso])
        for id_nodo in ids:
            if [label, id_nodo] not in entidades_de_la_pregunta:
                entidades_de_la_pregunta.append([label, id_nodo])
    entidades.append(entidades_de_la_pregunta)
    enlaces.append(enlaces_de_la_pregunta)
    sin_enlazar.append(menciones_sin_nodo)
    menciones.append(textos)
preguntas["menciones"] = menciones
preguntas["entidades"] = entidades
preguntas["enlaces"] = enlaces
preguntas["sin_enlazar"] = sin_enlazar
preguntas["con_mencion"] = preguntas["menciones"].str.len() > 0
preguntas["todas_enlazan"] = preguntas["con_mencion"] & (preguntas["sin_enlazar"].str.len() == 0)

## oro y su grupo

El oro se enlaza como en armar_conjuntos.ipynb. El grupo sale de los ítems con nodo sobre el total.

In [4]:
oro_nodos, oro_con_nodo = [], []
for pregunta in preguntas.itertuples():
    nodos_del_oro, con_nodo = [], 0
    for sinonimos in pregunta.oro:
        coincidencias = []
        for sinonimo in sinonimos:
            for tipo in TIPOS_DE_NODO:
                for id_nodo in linkeador.exacto(tipo, [sinonimo]):
                    if [tipo, id_nodo] not in coincidencias:
                        coincidencias.append([tipo, id_nodo])
        nodos_del_oro.append(coincidencias)
        con_nodo += bool(coincidencias)
    oro_nodos.append(nodos_del_oro)
    oro_con_nodo.append(con_nodo)
preguntas["oro_nodos"] = oro_nodos
preguntas["oro_con_nodo"] = oro_con_nodo
preguntas["n_oro"] = preguntas["oro"].str.len()
preguntas["fraccion_oro"] = preguntas["oro_con_nodo"] / preguntas["n_oro"]
#si un nodo de la respuesta es una entidad de la pregunta, entra al subgrafo por construccion y la seleccion no tiene
#que encontrarlo. en el grupo nada se cumple solo
preguntas["respuesta_no_es_entidad"] = [not any(nodo in entidades for item in nodos for nodo in item)
                                        for nodos, entidades in zip(preguntas["oro_nodos"], preguntas["entidades"])]


def grupo_del_oro(fraccion):
    "el grupo de una pregunta segun la fraccion de items del oro con nodo en el grafo entero"
    if fraccion == 0:
        return "nada"
    if fraccion < 0.5:
        return "menos_de_la_mitad"
    return "la_mitad_o_mas"


preguntas["grupo_oro"] = preguntas["fraccion_oro"].map(grupo_del_oro)

## cuanto se va perdiendo



In [5]:
CONDICIONES = [("1. Qwen marca alguna mención", preguntas["con_mencion"]),
               ("2. todas las menciones enlazan", preguntas["todas_enlazan"]),
               ("4. la respuesta no es una entidad de la pregunta", preguntas["respuesta_no_es_entidad"])]
quedan = pd.Series(True, index=preguntas.index)
filas = {"preguntas de list en 14b": len(preguntas)}
for nombre, condicion in CONDICIONES:
    quedan = quedan & condicion
    filas[nombre] = quedan.sum()
entran = preguntas[quedan]
display(pd.Series(filas).to_frame("preguntas"))

#por grupo: cuantas son, el largo de la lista de oro (mediana y cuartiles) y la fraccion media de items con nodo
largo_por_grupo = entran.groupby("grupo_oro")["n_oro"]
por_grupo = pd.DataFrame({"preguntas": largo_por_grupo.size(),
                          "largo_mediana": largo_por_grupo.median(),
                          "largo_q1": largo_por_grupo.quantile(0.25),
                          "largo_q3": largo_por_grupo.quantile(0.75),
                          "fraccion_media": entran.groupby("grupo_oro")["fraccion_oro"].mean()})
por_grupo = por_grupo.reindex(GRUPOS)
display(por_grupo.round(2))
#las que tienen todo el oro con nodo son las del conjunto de armar_conjuntos.ipynb, que caen en la_mitad_o_mas
print("con todo el oro con nodo:", (entran["fraccion_oro"] == 1).sum())
assert PREGUNTAS_POR_GRUPO <= por_grupo["preguntas"].min(), "PREGUNTAS_POR_GRUPO es mayor que el grupo mas chico"

,preguntas
preguntas de list en 14b,1130
1. Qwen marca alguna mención,786
2. todas las menciones enlazan,345
4. la respuesta no es una entidad de la pregunta,334


,preguntas,largo_mediana,largo_q1,largo_q3,fraccion_media
grupo_oro,,,,,
nada,103,4.0,2.0,5.0,0.00
menos_de_la_mitad,74,6.0,4.0,10.0,0.28
la_mitad_o_mas,157,4.0,3.0,7.0,0.78


con todo el oro con nodo: 51


## Ejemplos para revisar

Algunos sorteados de cada grupo, para ver a ojo cuánto falta en el grafo y cuánto es falla del enlace.

In [6]:
for grupo in GRUPOS:
    print(f"\n=== {grupo}")
    for pregunta in entran[entran["grupo_oro"] == grupo].sample(n=8, random_state=SEED).itertuples():
        #cada item del oro por su primer sinonimo, con + si tiene nodo y - si no
        items = [("+ " if nodos else "- ") + sinonimos[0] for sinonimos, nodos in zip(pregunta.oro, pregunta.oro_nodos)]
        print(f"- {pregunta.body}")
        print(f"    entidades de la pregunta: {[e[0] for e in pregunta.enlaces]}")
        print(f"    oro: {' | '.join(items)}")


=== nada
- Where can we find the protein lacritin?
    entidades de la pregunta: ['protein lacritin']
    oro: - lacrimal gland | - salivary glands | - thyroid | - tear fluid
- Which processes are affected by pathogenic SPTBN1 variants?
    entidades de la pregunta: ['SPTBN1']
    oro: - βII-spectrin stability | - Disrupt binding to key molecular partners | - Disturb cytoskeleton organization and dynamics
- What doses of fingolimod were administered during the FREEDOMS trial?
    entidades de la pregunta: ['fingolimod', 'FREEDOMS trial']
    oro: - 0.5mg | - 1.25mg
- Name the phase 3 clinical trials for tofacitinib in colitis.
    entidades de la pregunta: ['tofacitinib', 'colitis']
    oro: - OCTAVE Induction 1 | - OCTAVE Induction 2 | - OCTAVE Sustain
- Which bacteria cause diphtheria?
    entidades de la pregunta: ['diphtheria']
    oro: - Corynebacterium ulcerans | - Corynebacterium diphtheriae
- Which are the consequences of the hyperphosphorylated tau in Alzheimers' Disease?
   

In [7]:
nombre_de = linkeador.nombre_de

consultas = {}
for pregunta in entran.sort_values("id").itertuples():
    label, id_nodo = pregunta.entidades[0]
    base = re.sub(r"[^a-z0-9]+", "_", nombre_de[(label, id_nodo)].lower()).strip("_")
    nombre = base + "_" + pregunta.id[-6:]
    if nombre in consultas:
        nombre = base + "_" + pregunta.id
    assert nombre not in consultas, f"nombre repetido: {nombre}"
    consultas[nombre] = {"pregunta": pregunta.body, "entidades": pregunta.entidades, "id_bioasq": pregunta.id,
                         "tarea": "list", "enlaces": pregunta.enlaces, "sin_enlazar": pregunta.sin_enlazar,
                         "oro": pregunta.oro, "oro_nodos": pregunta.oro_nodos,
                         "grupo_oro": pregunta.grupo_oro, "fraccion_oro": pregunta.fraccion_oro,
                         "ideal_answer": pregunta.ideal_answer}
with open(f"consultas_{TAREA}.json", "w") as archivo:
    json.dump(consultas, archivo, indent=1, ensure_ascii=False)

#el sorteo es por grupo, sobre los nombres ordenados, asi que no depende del orden en que se armaron las consultas
elegidos = []
for grupo in GRUPOS:
    nombres = pd.Series(sorted(nombre for nombre, datos in consultas.items() if datos["grupo_oro"] == grupo))
    elegidos.extend(nombres.sample(n=PREGUNTAS_POR_GRUPO, random_state=SEED))
muestra = {nombre: consultas[nombre] for nombre in sorted(elegidos)}
with open(f"muestra_{TAREA}.json", "w") as archivo:
    json.dump(muestra, archivo, indent=1, ensure_ascii=False)

pd.DataFrame([{"consulta": consulta, "id_bioasq": datos["id_bioasq"], "pregunta": datos["pregunta"],
               "grupo_oro": datos["grupo_oro"], "fraccion_oro": round(datos["fraccion_oro"], 3),
               "menciones": " | ".join(f"{e[0]} -> {', '.join(nombre_de[(e[1], i)] for i in e[2])} [{e[3]}]" for e in datos["enlaces"]),
               "entidades": " | ".join(f"{label}:{i}" for label, i in datos["entidades"]),
               "respuesta": " || ".join(("+ " if nodos else "- ") + sinonimos[0] for sinonimos, nodos in zip(datos["oro"], datos["oro_nodos"])),
               "en_muestra": consulta in muestra}
              for consulta, datos in consultas.items()]).to_csv(f"preguntas_{TAREA}.csv", index=False)
print(f"consultas_{TAREA}.json", len(consultas), "   muestra", len(muestra))
pd.Series([datos["grupo_oro"] for datos in muestra.values()]).value_counts().reindex(GRUPOS).to_frame("en la muestra")

consultas_list_por_oro.json 334    muestra 75


,en la muestra
nada,25
menos_de_la_mitad,25
la_mitad_o_mas,25


## Contra la muestra de list ya corrida

Cuántas de esta muestra ya estaban en muestra_list.json. Reúsan el índice, pero las respuestas se vuelven a pedir.

In [8]:
with open("muestra_list.json") as archivo:
    corrida = {datos["id_bioasq"] for datos in json.load(archivo).values()}
en_las_dos = pd.Series([datos["grupo_oro"] for datos in muestra.values() if datos["id_bioasq"] in corrida])
en_las_dos.value_counts().reindex(GRUPOS).fillna(0).astype(int).to_frame("ya en muestra_list.json")

,ya en muestra_list.json
nada,0
menos_de_la_mitad,0
la_mitad_o_mas,7
